## 什么是TypedDict
TypedDict 是 Python 3.8+ 引入的一种类型提示工具，即带有类型声明的字典结构。适合需要快速定义
字典结构且无需 Pydantic 重量级功能的场景。

普通dict没有类型信息

In [ ]:
{
    "title": "xxx",
    "year": 2020,
    "director": 'xxx',
    "rating": 5
}

TypedDict可以进一步说明：
- 这个字典应该有哪些字段。
- 每个字段的类型是什么。
- TypedDict主要是类型声明，不是运行时强校验器。

In [ ]:
from typing_extensions import TypedDict, Annotated


class MovieDict(TypedDict):
    title: str
    year: int
    director: str
    rating: float

movie = MovieDict(
    title1="到门面韩国空军",
    year=2020,
    director="sss",
    rating=5
)

movie1: MovieDict = {
    "title": "xxx",
    "year": 2020,
    "director": "sss",
    "rating": 5
}

print(movie)
print(movie1)

实例化字典时给出的字段名称和TypedDict不完全一致，此时IDE的静态类型检查会标记，但不会导致运行时异常。

## 基本使用

### Annotated的使用
Annotated用来在类型之外，再附加一些额外信息，即元数据，类似Pydantic的Field

基本形式：
```
Annotated[类型, 附加信息1, 附加信息2, ...]
```

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考模式
)

### 返回简单结构

In [ ]:
from typing_extensions import Annotated

class MovieTypedDict(TypedDict):
    """电影详细信息"""
    title: Annotated[str, "电影的名称"]
    year: Annotated[int, "电影的公映年份，用4位数表示"]
    director: Annotated[str, "电影的导演"]
    rating: Annotated[float, "电影在10分制下的评分，可包含一位小数"]

struct_model = model.with_structured_output(MovieTypedDict)
response = struct_model.invoke("介绍电影星际穿越")
print(response)
print(type(response))

### 返回嵌套结构

In [ ]:
from typing import TypedDict, List, Annotated

# 使用TypedDict定义嵌套结构
class Actor(TypedDict):
    """演员情况"""
    name: Annotated[str, "演员姓名"]
    role: Annotated[str, "饰演的角色"]

class Movie(TypedDict):
    """电影情况"""
    title: Annotated[str, "电影标题"]
    year: Annotated[int, "上映年份"]
    director: Annotated[str, "导演"]
    cast: Annotated[List[Actor], "演员列表"] # 嵌套列表定义
    rating: Annotated[float, "评分"]

struct_model = model.with_structured_output(Movie)
response = struct_model.invoke("介绍电影星际穿越")
print(response)

### ...的使用
 ... 是Python的字面量，等价于 Ellipsis ，可以理解为占位符。下游框架（如LangChain）可以对 ... 作定制化处理，如LangChain中Annotated的 ... 表示当前字段是必须存在的，不可省略，用来指示模型的输出。

In [ ]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """
    电影的详细信息
    """
    title: Annotated[str, ..., "电影标题"]
    year: Annotated[int, ..., "电影上映年份"]
    director: Annotated[str, Ellipsis, "导演"]
    rating: Annotated[float, "电影评分，满分十分"]

struct_model = model.with_structured_output(MovieDict)
response = struct_model.invoke("根据这段话抽取盗梦空间的信息，不包含的信息必须留空：盗梦空间上映于2010年，我们并不知道它的导演是谁。")
print(response)